# Osonye Onyemazuwa — Week 2: Database Setup
### Day 9: Create the database (PostgreSQL); load cleaned Ad Spend data

**Issue #32:** Create the database (PostgreSQL/SQLite); load cleaned Ad Spend data.

**Note:** this notebook was written and schema-checked against the Day 8
design, but not executed against a live PostgreSQL server in this
environment (no server/network access here). Run this locally against
your own Postgres instance and flag anything that errors.

### Requirements
```
pip install psycopg2-binary sqlalchemy python-dotenv
```
You will need a running PostgreSQL server and a database created first, e.g.:
```
create db attribution
```

### Credentials setup (do this once, before running the cells below)
Create a `.env` file in this same folder (never commit this file) with:
```
DB_USER=postgres
DB_PASSWORD=your_actual_password
DB_HOST=localhost
DB_PORT=5432
DB_NAME=attribution
```
Then confirm `.env` is listed in `.gitignore` before running anything below.


In [ ]:
import sys
print(sys.executable)

In [ ]:
import sys
!{sys.executable} -m pip install psycopg2-binary

In [ ]:
import os
import urllib.parse
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Credentials loaded from .env (never hardcoded, never committed -- see .gitignore)
load_dotenv()

DB_USER = os.environ.get("DB_USER")
DB_PASSWORD = urllib.parse.quote_plus(os.environ.get("DB_PASSWORD")) # encloses special characters like @ 
DB_HOST = os.environ.get("DB_HOST")
DB_PORT = os.environ.get("DB_PORT")
DB_NAME = os.environ.get("DB_NAME")

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

print("Connected as:", DB_USER)  # safe to print -- never print DB_PASSWORD


## Create tables (schema from Day 8)

In [ ]:
create_campaigns = text("""
CREATE TABLE IF NOT EXISTS campaigns (
    campaign_id      INTEGER PRIMARY KEY,
    channel          TEXT NOT NULL,
    objective        TEXT,
    start_date       DATE,
    end_date         DATE,
    target_segment   TEXT,
    expected_uplift  NUMERIC
);
""")

create_ad_spend = text("""
CREATE TABLE IF NOT EXISTS ad_spend (
    spend_id       TEXT PRIMARY KEY,
    date           DATE NOT NULL,
    campaign_id    INTEGER NOT NULL REFERENCES campaigns(campaign_id),
    channel        TEXT NOT NULL,
    utm_source     TEXT,
    utm_medium     TEXT,
    utm_campaign   TEXT,
    impressions    INTEGER,
    clicks         INTEGER,
    ad_spend       NUMERIC NOT NULL,
    currency       TEXT
);
""")

with engine.begin() as conn:
    conn.execute(create_campaigns)
    conn.execute(create_ad_spend)

print("Tables created (or already existed).")


**Note on table order:** `ad_spend.campaign_id` has a `REFERENCES
campaigns(campaign_id)` foreign key, so `campaigns` must be created (and
loaded with data) before any rows can be inserted into `ad_spend`,
otherwise Postgres will reject the insert with a foreign key violation.


## Load cleaned data